In [1]:
import os

folder_path = "C:/Users/soura/OneDrive/Documents/Jupyter Notebooks/input_files/"

In [2]:
import pandas as pd

file_name = "Global_Ecommerce_Dataset.xlsx"
full_path = os.path.join(folder_path, file_name)

data = pd.read_excel(full_path,sheet_name="Summary")

In [3]:
data.head(10)    # NOT a good data consumption due to the table starting from row 5

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN
5,Row ID,Region,Total Revenue,Total Orders,Avg Order Value
6,0,Latin America,7599512.94,2065,3680.15
7,1,Europe,7592472,2056,3692.84
8,2,North America,7399986.45,1970,3756.34
9,3,Asia Pacific,7389265.25,1975,3741.4


In [4]:
# fix this by specifying the header row
data = pd.read_excel(full_path,sheet_name="Summary",header=6,index_col="Row ID")
data.head(10) 

,Region,Total Revenue,Total Orders,Avg Order Value
Row ID,,,,
0,Latin America,7599512.94,2065,3680.15
1,Europe,7592472.00,2056,3692.84
2,North America,7399986.45,1970,3756.34
3,Asia Pacific,7389265.25,1975,3741.40
4,Middle East,7316250.49,1934,3782.96


In [5]:
data.describe()   # provide some basic statistical analysis

,Total Revenue,Total Orders,Avg Order Value
count,5.000000e+00,5.000000,5.000000
mean,7.459497e+06,2000.000000,3730.738000
std,1.287249e+05,57.536945,43.277252
min,7.316250e+06,1934.000000,3680.150000
25%,7.389265e+06,1970.000000,3692.840000
50%,7.399986e+06,1975.000000,3741.400000
75%,7.592472e+06,2056.000000,3756.340000
max,7.599513e+06,2065.000000,3782.960000


In [6]:
def get_header_row(file_path, sheet_name=0, sample_rows=20):
    # Read the first chunk of the sheet without setting a header
    df_sample = pd.read_excel(file_path, sheet_name=sheet_name, header=None, nrows=sample_rows)
    
    # Count the number of non-null values in each row
    non_null_counts = df_sample.notna().sum(axis=1)
    
    # The header is typically the first row with the maximum number of columns filled
    header_index = non_null_counts.idxmax()
    
    return header_index

# Detect the starting Row and capture the table accordingly

In [7]:
import pandas as pd

# Create an ExcelFile object (opens the file once)
xls = pd.ExcelFile(full_path)

print("Available sheets:", xls.sheet_names)

for sheet in xls.sheet_names:
    print(f"\n--- Loading Sheet: {sheet} ---")

    # Detect the header row for each sheet
    detected_header = get_header_row(full_path, sheet_name=sheet)
    print(f"Header starts at index: {detected_header}")

    if detected_header is None:
        print(f"Could not detect a header in '{sheet}', skipping.")
        continue

    df = pd.read_excel(xls, sheet_name=sheet, header=detected_header, index_col=0)
    print(df.head())

Available sheets: ['Summary', 'Transactions Data', 'Customers Master']

--- Loading Sheet: Summary ---
Header starts at index: 6
               Region  Total Revenue  Total Orders  Avg Order Value
Row ID                                                             
0       Latin America     7599512.94          2065          3680.15
1              Europe     7592472.00          2056          3692.84
2       North America     7399986.45          1970          3756.34
3        Asia Pacific     7389265.25          1975          3741.40
4         Middle East     7316250.49          1934          3782.96

--- Loading Sheet: Transactions Data ---
Header starts at index: 0
                              Date  Customer ID         Region  \
Transaction ID                                                   
TXN-000001     2023-04-13 19:00:00         1003         Europe   
TXN-000002     2023-09-28 10:00:00         1177         Europe   
TXN-000003     2023-03-13 20:00:00         1745  North America 

In [8]:
# Setting sheet_name=None loads every sheet into a dictionary
all_sheets_dict = pd.read_excel(full_path, sheet_name=None, header=0, index_col=0)

# Loop through the dictionary to access each DataFrame
for sheet_name, df in all_sheets_dict.items():
    print(f"\n--- Sheet Name: {sheet_name} ---")
    print(df.head()) # Prints the first 5 rows of the dataframe


--- Sheet Name: Summary ---
    Unnamed: 1 Unnamed: 2 Unnamed: 3 Unnamed: 4
NaN        NaN        NaN        NaN        NaN
NaN        NaN        NaN        NaN        NaN
NaN        NaN        NaN        NaN        NaN
NaN        NaN        NaN        NaN        NaN
NaN        NaN        NaN        NaN        NaN

--- Sheet Name: Transactions Data ---
                              Date  Customer ID         Region  \
Transaction ID                                                   
TXN-000001     2023-04-13 19:00:00         1003         Europe   
TXN-000002     2023-09-28 10:00:00         1177         Europe   
TXN-000003     2023-03-13 20:00:00         1745  North America   
TXN-000004     2024-09-06 18:00:00         1258  North America   
TXN-000005     2023-08-03 10:00:00         1651   Asia Pacific   

                      Category  Quantity  Unit Price  Discount  Total Sales  
Transaction ID                                                               
TXN-000001              S

# Detect the starting Row and Column and capture the table accordingly

In [9]:
def get_header_position(raw, min_filled=2):
    """
    Find where the table starts in a sheet read with header=None.
    Returns (header_row, start_col) as 0-based positions, or (None, None).
    The header is the first row with at least `min_filled` non-empty cells,
    so a single title cell above the table is skipped.
    """
    for row_idx, row in raw.iterrows():
        if row.notna().sum() >= min_filled:
            start_col = row.first_valid_index()  # first non-empty column in that row
            return row_idx, start_col
    return None, None

In [10]:
xls = pd.ExcelFile(full_path)
print("Available sheets:", xls.sheet_names)

for sheet in xls.sheet_names:
    print(f"\n--- Loading Sheet: {sheet} ---")

    # Read the sheet raw, with no header, so nothing gets shifted
    raw = pd.read_excel(xls, sheet_name=sheet, header=None)

    header_row, start_col = get_header_position(raw)
    
    if header_row is None:
        print(f"No table found in '{sheet}', skipping.")
        continue
    
    print(f"Header at row index {header_row}, column index {start_col}")

    # Slice the table out of the raw data
    df = raw.iloc[header_row + 1:, start_col:].copy()
    df.columns = raw.iloc[header_row, start_col:]

    # Clean up: drop columns with no header and fully empty rows
    df = df.loc[:, df.columns.notna()]
    df = df.dropna(how="all")

    # Same as index_col=0: use the first column as the index
    df = df.set_index(df.columns[0])

    print(df.head())

Available sheets: ['Summary', 'Transactions Data', 'Customers Master']

--- Loading Sheet: Summary ---
Header at row index 6, column index 0
6              Region Total Revenue Total Orders Avg Order Value
Row ID                                                          
0       Latin America    7599512.94         2065         3680.15
1              Europe       7592472         2056         3692.84
2       North America    7399986.45         1970         3756.34
3        Asia Pacific    7389265.25         1975          3741.4
4         Middle East    7316250.49         1934         3782.96

--- Loading Sheet: Transactions Data ---
Header at row index 0, column index 0
0                              Date Customer ID         Region  \
Transaction ID                                                   
TXN-000001      2023-04-13 19:00:00        1003         Europe   
TXN-000002      2023-09-28 10:00:00        1177         Europe   
TXN-000003      2023-03-13 20:00:00        1745  North Ameri